In [ ]:
# Render choropleth map buat visualisasi distribusi gap score
fig, ax = plt.subplots(figsize=(14, 10))

merged.plot(
    column="gap_score_composite",
    cmap="RdYlGn_r",
    legend=True,
    legend_kwds={"label": "Gap Score Composite", "shrink": 0.8},
    ax=ax,
    edgecolor="black",
    linewidth=0.5,
)

ax.set_title("Gap Score Distribusi Fasilitas Publik — Jawa Barat", fontsize=16, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()

# Export plot ke local directory
import os
output_dir = "./outputs"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "gap_score_map.png")
plt.savefig(output_path, dpi=150, bbox_inches="tight")
print(f"✅ Map saved to {output_path}")
plt.show()

In [ ]:
# Sanity check buat mastiin gap score valid
print("📊 Gap Score Summary Statistics:")
print(merged[["gap_score_composite", "gap_demografi", "gap_kesehatan"]].describe())

print("\n🔍 Sample kecamatan dengan gap score tertinggi:")
top_gaps = merged.nlargest(5, "gap_score_composite")[
    ["kecamatan_std", "gap_score_composite", "gap_demografi", "gap_kesehatan"]
]
print(top_gaps)

print("\n⚠️  Kecamatan dengan data tidak lengkap:")
missing = merged[merged["gap_score_composite"].isna()][["kecamatan_std"]]
if len(missing) > 0:
    print(missing)
else:
    print("✅ Semua kecamatan memiliki gap_score_composite")

In [ ]:
# Join data tabular & geodata via kecamatan_std
merged = gdf.merge(df, left_on="kecamatan_std", right_on="kecamatan_std", how="left")

matched = merged["gap_score_composite"].notna().sum()
total = len(merged)

print(f"✅ Merge selesai")
print(f"Matched: {matched} / {total} kecamatan ({100*matched/total:.1f}%)")
print(f"\nColumns setelah merge: {len(merged.columns)}")
print(f"\nData types:")
print(merged[["kecamatan_std", "gap_score_composite", "gap_demografi", "gap_kesehatan"]].dtypes)

In [ ]:
# Fetch GeoJSON batas kecamatan dari raw-data container
blob = blob_client.get_blob_client("raw-data", "batas_kecamatan_jabar.geojson")
geo_data = blob.download_blob().readall()
gdf = gpd.read_file(io.BytesIO(geo_data))

print(f"✅ Loaded {len(gdf)} features (kecamatan boundaries)")
print(f"\nCRS: {gdf.crs}")
print(f"Columns: {list(gdf.columns)}")
print(f"\nFirst 3 features:")
gdf.head(3)

In [ ]:
# Load data master dari processed-data container
df = read_blob_csv("processed-data", "master_kecamatan.csv")
print(f"✅ Loaded {len(df)} kecamatan")
print(f"\nColumns: {list(df.columns)}")
print(f"\nFirst 5 rows:")
df.head()

In [ ]:
def read_blob_csv(container_name, blob_name):
    """
    Fetch CSV dari Blob Storage langsung ke memory (bypass disk I/O).
    
    Args:
        container_name: Nama container blob
        blob_name: Nama file blob
        
    Returns:
        pandas.DataFrame dengan data CSV
    """
    blob = blob_client.get_blob_client(container_name, blob_name)
    data = blob.download_blob().readall()
    return pd.read_csv(io.BytesIO(data))

print("✅ Helper function ready")

In [ ]:
import os
import io
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from azure.storage.blob import BlobServiceClient

# Setup connection ke Azure Blob Storage
connection_string = os.environ.get("AZURE_STORAGE_CONNECTION_STRING")
if not connection_string:
    raise ValueError("AZURE_STORAGE_CONNECTION_STRING environment variable harus di-set")

blob_client = BlobServiceClient.from_connection_string(connection_string)
print("✅ Connected to Azure Blob Storage")

# GapMap Jabar — EDA di Azure ML Studio

Notebook ini nge-load data GapMap Jabar dari Azure Blob Storage:
- Data tabular: `master_kecamatan.csv` 
- Data geospasial: `batas_kecamatan_jabar.geojson`

Terus dilanjut sama sanity check & render visualisasi distribusi gap score se-Jawa Barat.